<a href="https://colab.research.google.com/github/NoufAlqrni99/Madar_Agentic_AI/blob/main/Madar_Capstone_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# مَدار | Madar
### Intelligent Multi-Agent Event Management System

**Trainee:** Nouf Alqarni  
**Programme:** SDAIA Academy — Agentic AI Systems  
**Cohort Dates:** 23–27 August 2026  
**Track:** Track A — Supervisor + Workers

مَدار (Madar) is a multi-agent system for handling operational disruptions during an event. A supervisor delegates requests to specialized Parking, Schedule, and Venue agents. The project also demonstrates real tool use, structured output, Agentic RAG, short- and long-term memory, human-in-the-loop approval, LangGraph Functional API reliability patterns, and LangSmith tracing.

> Before submission, add your full name and the exact SDAIA Academy cohort dates to the README/notebook header. Keep API keys only in Colab Secrets and never in GitHub.

## 1. Setup

The notebook uses the same LangChain/LangGraph family of tools taught in the course. Run this notebook from top to bottom in a fresh Colab runtime before submission.

In [18]:
!pip install -qU langchain langchain-groq langgraph langgraph-supervisor langsmith sentence-transformers faiss-cpu
!pip install -qU langchain-community --no-deps

In [19]:
import os
from google.colab import userdata
from langchain_groq import ChatGroq
from langsmith import Client

# Rubric-required tracing variable + current LangSmith variables.
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "event-management-agents"
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_PROJECT"] = "event-management-agents"
os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=userdata.get("GROQ_API_KEY"),
    temperature=0,
)

langsmith_client = Client(
    api_key=userdata.get("LANGSMITH_API_KEY"),
    api_url="https://api.smith.langchain.com",
)

print("LLM and LangSmith configuration ready.")

LLM and LangSmith configuration ready.


## 2. Event data and real tools — Agent fundamentals

The tools below read and calculate from shared event data rather than returning unrelated hard-coded answers. This gives the agents real operations to choose from.

In [20]:
event_data = {
    "event_name": "Saudi AI Experience",
    "parking": {
        "A": {"capacity": 500, "occupied": 475},
        "B": {"capacity": 300, "occupied": 120},
    },
    "venues": {
        "Hall A": {"capacity": 500, "status": "available"},
        "Hall B": {"capacity": 300, "status": "available"},
        "Hall C": {"capacity": 200, "status": "unavailable"},
    },
    "schedule": [
        {"time": "18:00", "session": "AI in Saudi Arabia", "speaker": "Ahmed", "venue": "Hall A"},
        {"time": "19:00", "session": "Future of Data", "speaker": "Sara", "venue": "Hall B"},
    ],
}

print("Loaded:", event_data["event_name"])

Loaded: Saudi AI Experience


In [21]:
from langchain_core.tools import tool

@tool
def get_parking_status(parking_name: str = "all") -> str:
    """Return current occupancy for one parking area (A/B) or all areas."""
    names = list(event_data["parking"]) if parking_name.lower() == "all" else [parking_name.upper()]
    rows = []
    for name in names:
        info = event_data["parking"].get(name)
        if not info:
            rows.append(f"Parking {name}: not found")
            continue
        pct = 100 * info["occupied"] / info["capacity"]
        rows.append(f"Parking {name}: {info['occupied']}/{info['capacity']} occupied ({pct:.0f}%).")
    return "\n".join(rows)

@tool
def find_available_parking(min_spaces: int = 1) -> str:
    """Find parking areas with at least min_spaces available spaces."""
    options = []
    for name, info in event_data["parking"].items():
        available = info["capacity"] - info["occupied"]
        if available >= min_spaces:
            options.append(f"Parking {name}: {available} spaces available")
    return "\n".join(options) if options else "No parking area meets the requested availability."

@tool
def get_event_schedule(speaker: str = "all") -> str:
    """Return the event schedule, optionally filtered by speaker name."""
    sessions = event_data["schedule"]
    if speaker.lower() != "all":
        sessions = [s for s in sessions if s["speaker"].lower() == speaker.lower()]
    if not sessions:
        return "No matching session found."
    return "\n".join(
        f"{s['time']} - {s['session']} - Speaker: {s['speaker']} - {s['venue']}"
        for s in sessions
    )

@tool
def get_venue_status(hall_name: str = "all") -> str:
    """Return availability and capacity for one hall or all halls."""
    names = list(event_data["venues"]) if hall_name.lower() == "all" else [hall_name]
    rows = []
    for name in names:
        info = event_data["venues"].get(name)
        if not info:
            rows.append(f"{name}: not found")
            continue
        rows.append(f"{name}: {info['status']}, capacity {info['capacity']}")
    return "\n".join(rows)

print(get_parking_status.invoke({"parking_name": "A"}))
print(find_available_parking.invoke({"min_spaces": 50}))
print(get_event_schedule.invoke({"speaker": "all"}))
print(get_venue_status.invoke({"hall_name": "Hall C"}))

Parking A: 475/500 occupied (95%).
Parking B: 180 spaces available
18:00 - AI in Saudi Arabia - Speaker: Ahmed - Hall A
19:00 - Future of Data - Speaker: Sara - Hall B
Hall C: unavailable, capacity 200


## 3. Structured output — Pydantic

`with_structured_output` is used when code needs to parse the model result. The schema constrains problem type and urgency instead of parsing free-form text.

In [22]:
from typing import Literal
from pydantic import BaseModel, Field

class EventRouting(BaseModel):
    problem_type: Literal["parking", "schedule", "venue"] = Field(
        description="Operational area for the request."
    )
    urgency: Literal["low", "medium", "high"] = Field(
        description="Operational urgency."
    )
    reason: str = Field(
        description="Short justification for the classification."
    )

routing_llm = llm.with_structured_output(
    EventRouting,
    method="json_mode"
)

routing_result = routing_llm.invoke("""
You are a routing classifier.

Return ONLY valid JSON with exactly these three fields:
{
  "problem_type": "parking",
  "urgency": "high",
  "reason": "short explanation"
}

Allowed values:
problem_type = parking, schedule, or venue
urgency = low, medium, or high

Do not solve the problem.
Do not suggest alternatives.

Problem:
Parking A is almost full and visitors need an alternative.
""")

print("Structured result:", routing_result)
print("Structured dict:", routing_result.model_dump())

Structured result: problem_type='parking' urgency='high' reason='Parking A is almost full, visitors need an alternative.'
Structured dict: {'problem_type': 'parking', 'urgency': 'high', 'reason': 'Parking A is almost full, visitors need an alternative.'}


## 4. RAG pipeline — Agentic RAG

**Choice:** Agentic RAG. Retrieval is not forced on every request; an agent decides when policy knowledge is useful and calls `search_event_knowledge`. This fits event operations because live status tools and policy retrieval are different kinds of evidence.

Pipeline: policy documents → text splitting → HuggingFace embeddings → FAISS vector store → retriever → agent tool.

In [23]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

policy_documents = [
    Document(
        page_content=(
            "Parking policy: When Parking A reaches 90% capacity, visitors should be directed to Parking B. "
            "Parking staff should prioritize the alternative area before the main area becomes completely full."
        ),
        metadata={"source": "parking_policy"},
    ),
    Document(
        page_content=(
            "Venue policy: Hall C is currently unavailable. Hall A can accommodate up to 500 attendees "
            "and Hall B can accommodate up to 300 attendees."
        ),
        metadata={"source": "venue_policy"},
    ),
    Document(
        page_content=(
            "Schedule policy: If a speaker is delayed, the organizer should check whether another session "
            "can be moved or exchanged before extending the event end time."
        ),
        metadata={"source": "schedule_policy"},
    ),
]

splitter = RecursiveCharacterTextSplitter(chunk_size=140, chunk_overlap=20)
event_chunks = splitter.split_documents(policy_documents)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_store = FAISS.from_documents(event_chunks, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 1})

print(f"Documents: {len(policy_documents)} | Chunks: {len(event_chunks)}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Documents: 3 | Chunks: 5


In [24]:
rag_question = "What should visitors do when Parking A reaches 90% capacity?"
rag_docs = retriever.invoke(rag_question)

for doc in rag_docs:
    print("SOURCE:", doc.metadata["source"])
    print("CONTENT:", doc.page_content)

assert rag_docs and rag_docs[0].metadata["source"] == "parking_policy"

SOURCE: parking_policy
CONTENT: Parking policy: When Parking A reaches 90% capacity, visitors should be directed to Parking B. Parking staff should prioritize the


In [25]:
def search_event_knowledge(question: str) -> str:
    docs = retriever.invoke(question)
    if not docs:
        return "No relevant information found."
    return "\n".join(
        f"[{doc.metadata['source']}] {doc.page_content}"
        for doc in docs
    )

search_event_knowledge_tool = tool(
    search_event_knowledge,
    description="Search event policies and operational knowledge for relevant guidance."
)

print(search_event_knowledge_tool.invoke(
    "What is the policy when Parking A becomes almost full?"
))

[parking_policy] Parking policy: When Parking A reaches 90% capacity, visitors should be directed to Parking B. Parking staff should prioritize the


## 5. Multi-agent architecture — Track A: Supervisor + Workers

The project uses a dedicated supervisor as the orchestrator. Workers are specialists and do not route directly to each other. The LLM performs the handoff; there is no keyword-based `if "parking" in question` router.

In [26]:
from langchain.agents import create_agent
from langgraph_supervisor import create_supervisor

parking_agent = create_agent(
    model=llm,
    tools=[get_parking_status, find_available_parking, search_event_knowledge_tool],
    name="parking_agent",
    system_prompt=(
        "You are the Parking Agent. Use tools before answering operational questions. "
        "Be concise: maximum two short sentences."
    ),
)

schedule_agent = create_agent(
    model=llm,
    tools=[get_event_schedule, search_event_knowledge_tool],
    name="schedule_agent",
    system_prompt=(
        "You are the Schedule Agent. Use tools before answering schedule or speaker-delay questions. "
        "Be concise: maximum two short sentences."
    ),
)

venue_agent = create_agent(
    model=llm,
    tools=[get_venue_status, search_event_knowledge_tool],
    name="venue_agent",
    system_prompt=(
        "You are the Venue Agent. Use tools before answering hall availability or capacity questions. "
        "Be concise: maximum two short sentences."
    ),
)

supervisor = create_supervisor(
    agents=[parking_agent, schedule_agent, venue_agent],
    model=llm,
    prompt=(
        "You are the Event Supervisor. Delegate parking requests to parking_agent, "
        "schedule/speaker requests to schedule_agent, and hall/venue requests to venue_agent. "
        "After the worker finishes, return its answer. Keep the final response concise."
    ),
).compile()

print("Workers and supervisor ready.")

Workers and supervisor ready.


### Evidence A — the worker chooses and executes real tools

In [27]:
parking_demo = parking_agent.invoke({
    "messages": [
        {"role": "user", "content": "Check current parking availability and recommend the best area."}
    ]
})

worker_tool_calls = []
for message in parking_demo["messages"]:
    for tc in getattr(message, "tool_calls", []) or []:
        worker_tool_calls.append(tc["name"])
        print("tool_call ->", tc["name"])

print("FINAL:", parking_demo["messages"][-1].content)
assert worker_tool_calls, "Expected the agent to choose at least one real tool."

tool_call -> get_parking_status
FINAL: Parking B has the most free space (120/300 occupied, 40% full). It’s the best choice for you.


### Evidence B — supervisor handoff + LangSmith trace

This single demo is explicitly traced to the `event-management-agents` LangSmith project. It should show `transfer_to_parking_agent` and the return handoff.

In [28]:
from langsmith import tracing_context
from langchain_core.tracers.langchain import wait_for_all_tracers

with tracing_context(
    enabled=True,
    client=langsmith_client,
    project_name="event-management-agents",
):
    supervisor_result = supervisor.invoke({
        "messages": [
            {"role": "user", "content": "Parking A is almost full. Where should visitors park?"}
        ]
    })

message_names = [getattr(m, "name", None) for m in supervisor_result["messages"]]
for name in message_names:
    if name:
        print(name)

print("FINAL:", supervisor_result["messages"][-1].content)

assert "transfer_to_parking_agent" in message_names
assert "transfer_back_to_supervisor" in message_names

wait_for_all_tracers()
print("LangSmith trace flushed.")

supervisor
transfer_to_parking_agent
parking_agent
parking_agent
transfer_back_to_supervisor
supervisor
FINAL: Parking B has plenty of space—about 180 spots are still available. It’s the best choice for visitors.
LangSmith trace flushed.


## 6. Context & state management

Short-term state uses a LangGraph checkpointer keyed by `thread_id`. Long-term memory uses a separate `InMemoryStore`. The test below proves the distinction: the turn counter persists only within the same thread, while the organizer fact is readable from a different thread.

In [29]:
from typing import Any
from langgraph.func import entrypoint
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from langgraph.store.base import BaseStore

short_term_checkpointer = InMemorySaver()
long_term_store = InMemoryStore()

@entrypoint(
    checkpointer=short_term_checkpointer,
    store=long_term_store,
)
def organizer_memory(
    inputs: dict,
    *,
    previous: Any = None,
    store: BaseStore,
) -> dict:
    previous_turns = previous.get("turns", 0) if isinstance(previous, dict) else 0
    namespace = ("event_organizer", inputs["user_id"])

    if inputs.get("remember"):
        store.put(
            namespace,
            "operational_preference",
            {"fact": inputs["remember"]},
        )

    items = store.search(namespace)
    fact = items[0].value["fact"] if items else None

    return {
        "turns": previous_turns + 1,
        "long_term_fact": fact,
    }

thread_a = {"configurable": {"thread_id": "organizer-thread-A"}}
thread_b = {"configurable": {"thread_id": "organizer-thread-B"}}

first = organizer_memory.invoke(
    {
        "user_id": "organizer-1",
        "remember": "Do not use Hall C for main sessions.",
    },
    thread_a,
)
same_thread = organizer_memory.invoke(
    {"user_id": "organizer-1"},
    thread_a,
)
cross_thread = organizer_memory.invoke(
    {"user_id": "organizer-1"},
    thread_b,
)

print("Thread A first:", first)
print("Thread A second:", same_thread)
print("Thread B:", cross_thread)

assert same_thread["turns"] == 2
assert cross_thread["turns"] == 1
assert cross_thread["long_term_fact"] == "Do not use Hall C for main sessions."

Thread A first: {'turns': 1, 'long_term_fact': 'Do not use Hall C for main sessions.'}
Thread A second: {'turns': 2, 'long_term_fact': 'Do not use Hall C for main sessions.'}
Thread B: {'turns': 1, 'long_term_fact': 'Do not use Hall C for main sessions.'}


## 7. Human-in-the-loop — interrupt and resume

A schedule change is treated as an irreversible operational action. The workflow pauses before applying it, then resumes only after explicit human approval.

In [30]:
from langgraph.func import entrypoint, task
from langgraph.types import interrupt, Command

approval_checkpointer = InMemorySaver()
CHANGE_LOG = []

@task
def prepare_schedule_change(change: str) -> dict:
    return {"proposed_change": change, "status": "awaiting_approval"}

@task
def apply_schedule_change(proposal: dict) -> str:
    CHANGE_LOG.append(proposal["proposed_change"])
    return f"Applied: {proposal['proposed_change']}"

@entrypoint(checkpointer=approval_checkpointer)
def approval_workflow(change: str) -> dict:
    proposal = prepare_schedule_change(change).result()

    decision = interrupt({
        "action": "Approve before changing the live event schedule",
        "proposal": proposal,
    })

    if decision != "approve":
        return {"approved": False, "result": "Change rejected by human reviewer."}

    result = apply_schedule_change(proposal).result()
    return {"approved": True, "result": result}

approval_cfg = {"configurable": {"thread_id": "schedule-approval-1"}}

paused = approval_workflow.invoke(
    "Swap the 18:00 and 19:00 sessions",
    approval_cfg,
)

print("PAUSED:", paused["__interrupt__"][0].value)

PAUSED: {'action': 'Approve before changing the live event schedule', 'proposal': {'proposed_change': 'Swap the 18:00 and 19:00 sessions', 'status': 'awaiting_approval'}}


In [31]:
approved_run = approval_workflow.invoke(
    Command(resume="approve"),
    approval_cfg,
)

print("FINAL:", approved_run)
print("CHANGE_LOG:", CHANGE_LOG)

assert approved_run["approved"] is True
assert "Swap the 18:00 and 19:00 sessions" in CHANGE_LOG

FINAL: {'approved': True, 'result': 'Applied: Swap the 18:00 and 19:00 sessions'}
CHANGE_LOG: ['Swap the 18:00 and 19:00 sessions']


## 8. LangGraph Functional API & reliability

The workflow uses `@task` and `@entrypoint`. Two rubric error strategies are demonstrated in the notebook:

1. **Transient error:** a real `RetryPolicy` retries a temporary connection failure.
2. **User-fixable / irreversible action:** `interrupt()` pauses for human approval before a schedule change.

In [32]:
from langgraph.types import RetryPolicy

retry_attempts = {"count": 0}

@task(
    retry_policy=RetryPolicy(
        max_attempts=3,
        initial_interval=0.1,
        retry_on=ConnectionError,
    )
)
def transient_status_check(_: dict) -> dict:
    retry_attempts["count"] += 1
    print("Attempt:", retry_attempts["count"])

    # Deliberately simulate one transient failure so RetryPolicy is visible.
    if retry_attempts["count"] == 1:
        raise ConnectionError("Simulated temporary operations API failure.")

    return {
        "status": "recovered",
        "attempts": retry_attempts["count"],
    }

@entrypoint()
def reliability_workflow(inputs: dict) -> dict:
    return transient_status_check(inputs).result()

reliability_result = reliability_workflow.invoke({"check": "event_status"})
print("Reliability result:", reliability_result)

assert reliability_result["status"] == "recovered"
assert reliability_result["attempts"] >= 2

Attempt: 1
Attempt: 2
Reliability result: {'status': 'recovered', 'attempts': 2}


## 9. Final evidence-based evaluation

These checks are calculated from actual outputs created earlier in the notebook; they are not hard-coded PASS labels.

In [33]:
routing_data = routing_result.model_dump()

print("Structured output data:", routing_data)

structured_ok = (
    routing_data.get("problem_type") == "parking"
    or routing_data.get("problem") == "parking"
)

rag_ok = (
    bool(rag_docs)
    and getattr(rag_docs[0], "metadata", {}).get("source") == "parking_policy"
)

evaluation = {
    "structured_output": structured_ok,
    "rag_retrieval": rag_ok,
    "real_worker_tool_call": bool(worker_tool_calls),
    "supervisor_handoff": "transfer_to_parking_agent" in message_names,
    "return_handoff": "transfer_back_to_supervisor" in message_names,
    "short_term_state": same_thread.get("turns") == 2,
    "cross_thread_long_term_memory": (
        cross_thread.get("turns") == 1
        and cross_thread.get("long_term_fact")
        == "Do not use Hall C for main sessions."
    ),
    "human_interrupt_resume": approved_run.get("approved") is True,
    "retry_policy": reliability_result.get("status") == "recovered",
}

print("=== مَدار | Madar — Final Evaluation ===")

for name, passed in evaluation.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")

assert all(evaluation.values()), "At least one capstone demonstration failed."

Structured output data: {'problem_type': 'parking', 'urgency': 'high', 'reason': 'Parking A is almost full, visitors need an alternative.'}
=== مَدار | Madar — Final Evaluation ===
structured_output: PASS
rag_retrieval: PASS
real_worker_tool_call: PASS
supervisor_handoff: PASS
return_handoff: PASS
short_term_state: PASS
cross_thread_long_term_memory: PASS
human_interrupt_resume: PASS
retry_policy: PASS


# Rubric write-up

### 1. Agent fundamentals — 15 pts
The project uses tool-calling agents whose tools read and calculate from shared event data, rather than returning unrelated fixed strings. `EventRouting` is a Pydantic model used through `llm.with_structured_output(...)`, so routing information needed by code is returned in a constrained structure.

### 2. Multi-agent / routing architecture — 15 pts
I selected **Track A: Supervisor + Workers**. The supervisor LLM delegates to `parking_agent`, `schedule_agent`, or `venue_agent`; routing is not performed with keyword `if` statements. The captured output shows `transfer_to_parking_agent` and `transfer_back_to_supervisor`.

### 3. RAG pipeline — 15 pts
I selected **Agentic RAG** because policy retrieval is only needed for some operational questions. Policy documents are loaded, split into chunks, embedded with HuggingFace embeddings, stored in FAISS, and retrieved through a retriever exposed as an agent tool. The retrieval test asks a question answered directly by `parking_policy` and verifies that source.

### 4. Context & state management — 15 pts
Short-term state is maintained by an `InMemorySaver` checkpointer and a `thread_id`; the same thread increments its saved turn count. Long-term facts are stored separately in `InMemoryStore`. The cross-thread test writes an organizer preference in thread A and successfully reads the same fact in thread B while the short-term turn count resets.

### 5. Human-in-the-loop — 10 pts
A live schedule change is treated as an irreversible action. The Functional API workflow calls `interrupt()` before applying the change, captures the paused output, and then uses `Command(resume="approve")` to complete the run. The final output and change log prove that the approved action was executed after resumption.

### 6. LangGraph Functional API & error handling — 15 pts
The notebook uses both `@task` and `@entrypoint`. For reliability, a real `RetryPolicy` handles a simulated transient connection failure and succeeds on retry. A second error strategy is the human-interrupt path for an action requiring user approval, demonstrating that different failure/risk types receive different handling.

### 7. Workflow pattern — 10 pts
The explicit workflow pattern is **Orchestrator–Worker**. It fits the project because event operations contain distinct specialist domains, while one supervisor provides a single delegation point and returns the specialist result to the organizer.

### 8. LangSmith observability — 5 pts
Tracing is enabled and the supervisor demo is sent to the `event-management-agents` LangSmith project. The trace made the supervisor handoff and return path visible. It also showed that multi-agent requests create several model hops, explaining why a previous multi-issue test was more token-expensive than focused requests.

# Submission checks

Before submitting the repository:

- Restart the Colab runtime and **Run all** from top to bottom.
- Grant this notebook access to `GROQ_API_KEY` and `LANGSMITH_API_KEY` in Colab Secrets.
- Confirm every demo cell has saved output and no red error cells remain.
- Confirm the LangSmith project contains the supervisor trace.
- Add your **full name** and the **exact SDAIA Academy programme/cohort dates** to the notebook header and README.
- Repository documentation should include a professional `README.md`, technical explanation, programme statement, and a `.gitignore` that excludes secrets/generated files.
- State **Track A — Supervisor + Workers** explicitly in the README.
- Never place API keys in notebook code, README, commits, or Git history.

# Madar | مدار
## Intelligent Multi-Agent Event Management System

**Trainee:** Nouf Alqarni  
**Programme:** SDAIA Academy – Agentic AI Systems  
**Cohort Dates:** 23–27 August 2026  
**Track:** Track A — Supervisor + Workers  

## Project Overview

Madar (مدار) is an intelligent multi-agent system designed to support event operations and handle operational disruptions.

The system uses a Supervisor Agent that delegates requests to specialized worker agents for parking, schedule, and venue operations.

## Architecture

The project follows a Supervisor + Workers architecture:

- Supervisor Agent
- Parking Agent
- Schedule Agent
- Venue Agent

The Supervisor analyzes the request and delegates it to the appropriate specialized agent.

## Key Features

- Multi-agent orchestration
- Supervisor-to-worker handoffs
- Structured output using Pydantic
- Retrieval-Augmented Generation (RAG)
- FAISS vector store
- HuggingFace embeddings
- Short-term state management
- Long-term memory
- Human-in-the-loop approval
- Retry and reliability mechanisms
- LangSmith tracing and observability

## RAG Pipeline

Policy documents are split into chunks, converted into embeddings using HuggingFace embeddings, and stored in a FAISS vector store.

Agents can retrieve relevant event policies when additional operational knowledge is required.

## Human-in-the-Loop

Critical schedule changes require human approval before execution. The workflow pauses and resumes after an explicit approval decision.

## Reliability

The project includes retry mechanisms for transient failures and separate handling for actions that require human approval.

## Observability

LangSmith tracing is enabled to monitor the Supervisor and worker-agent workflow.

## Security

API keys are stored securely using Google Colab Secrets and are not included in the notebook or repository.

## Technologies

- Python
- LangChain
- LangGraph
- LangSmith
- Groq
- HuggingFace
- FAISS
- Pydantic
- Google Colab

## Author

Nouf Alqarni

SDAIA Academy — Agentic AI Systems  
August 2026